> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board, uploads the
> package and builds the bench on I2C bus 1. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host
>   against fake buses, writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

# Hardware and configuration

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | `sim.bus.FakeBus`, built through `bus_factory` | **bus 1** (scl=15 sda=16), built by the manager |
| **Mux** | `FakeMux` at `0x71` | SparkFun Qwiic Mux (TCA9548A) at `0x70` |
| **Parts** | `FakeSensirion` answering as an SHTC3 at `0x70`, `FakeRegisterPart` answering as a TMAG5273 at `0x35` on channel 3 | TMP102 at `0x48` on channel 2, TMAG5273 (B1) at `0x22` on channel 0 (README "The bench", 2026-11-14) |
| **Result** | not run yet | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): `I2CManager` takes an optional `bus_factory(bus_id, bus_cfg, freq)`. Without
one it builds `machine.I2C` as before; with one it builds whatever the factory returns. The
simulated bench (`sim.bench`) passes a factory that returns `FakeBus`es with the fake parts
attached, so a whole project runs on the host through the same manager code as on the board.

# Python file
location: `src/micropython_i2c/manager.py`

tests: `tests/test_manager.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/manager.py
"""``I2CManager`` builds live objects from a validated ``RoutingTable``; ``scan_check`` probes.

Build order: buses (one lock each, clocked to suit every part on them), then mux instances,
then device connections and instances. The clock is settled before any driver exists, so a
driver's init-time identity read runs on a safe bus.

Typical use::

    table = RoutingTable.loads(config_json).validate(registry())
    mgr = I2CManager(table, registry(), "feathers3")
    mgr.buses_build()
    ok, _ = mgr.scan()
    mgr.build()
    hall = mgr["hall"]

On the host, pass ``bus_factory`` to build fake buses instead of ``machine.I2C``
(``sim.bench.SimBench`` does this).

Generated by ``dev_notebooks/manager_devNB``: edit the notebook, not this file.
"""

# Imports
## External
`machine`, imported inside `buses_build` only when no `bus_factory` is given, so the module
imports on the host.

## Internal
`board_get` for the pin map, `default_lock` for the per-bus locks, and the connection layer.

In [ ]:
%%extract ../src/micropython_i2c/manager.py -a
from micropython_i2c.boards import board_get
from micropython_i2c.compat import default_lock
from micropython_i2c.connection import I2CConnection, path_release, path_select

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_manager.py
import pytest

from micropython_i2c import CommandI2CBase, NullLock
from micropython_i2c.devices import registry
from micropython_i2c.manager import I2CManager, scan_check
from micropython_i2c.routing import RoutingTable
from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart, FakeSensirion

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

from micropython_i2c import CommandI2CBase, NullLock
from micropython_i2c.devices import registry
from micropython_i2c.routing import RoutingTable
from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart, FakeSensirion

# Intent
**Purpose:** turn a validated routing table into live objects: one bus and one lock per bus
id, a driver instance per mux, then a connection and a driver instance per device.

```mermaid
flowchart LR
    T[RoutingTable] --> M[I2CManager]
    B[board pin map] --> M
    R[driver registry] --> M
    M -->|buses_build| BUS["machine.I2C, or bus_factory(...)"]
    M -->|muxes_build| MUX[TCA9548A ...]
    M -->|devices_build| DEV["I2CConnection + driver"]
```

**Bus clock:** the lowest `BAUD_MAX` of the board's bus entry and every part on that bus, so a
400 kHz part on a 1 MHz board bus runs the bus at 400 kHz. If some part's `BAUD_MIN` is above
that, the table cannot work and `bus_freq` says so.

**`scan_check`:** before building drivers, check every declared address answers on its
segment (select the path, scan, release). An answer means *something* is there; the driver's
identity read confirms what.

**Not its job:** validating the table (`RoutingTable.validate`), or talking to the parts
beyond building them.

## What the module defines

| Name | What it is |
|---|---|
| `scan_check` | check every declared address answers on its segment |
| `I2CManager` | builds buses (or fakes, via `bus_factory`), muxes and devices; `mgr["name"]` |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/manager.py -a
__all__ = ["I2CManager", "scan_check"]

## A fake topology for the cells below
One bus: a mux at `0x71`, an SHTC3 stand-in at `0x70` on the root segment (a `FakeSensirion`
whose one word is the SHTC3 ID), and a TMAG5273 stand-in at `0x35` behind mux channel 3 (a
register file holding the TMAG5273's ID registers). The config names the same three.

In [ ]:
%%extract ../tests/test_manager.py -a
REG = registry(["TCA9548A", "SHTC3", "TMAG5273"])


def config(**overrides):
    """The table the fake topology implements."""
    cfg = {
        "buses": [0],
        "muxes": {"mux_a": {"bus": 0, "address": "0x71", "driver": "TCA9548A"}},
        "devices": [
            {"name": "humidity", "bus": 0, "address": "0x70", "driver": "SHTC3"},
            {"name": "hall", "bus": 0, "address": "0x35", "driver": "TMAG5273",
             "path": [["mux_a", 3]]},
        ],
    }  # fmt: skip
    cfg.update(overrides)
    return cfg


def fake_topology():
    """A FakeBus wired as ``config()`` says."""
    bus = FakeBus()
    bus.attach(0x71, FakeMux())
    bus.attach(0x70, FakeSensirion([0x0807], CommandI2CBase.crc))
    bus.attach(0x35, FakeRegisterPart({0x0E: 0x49, 0x0F: 0x54, 0x0D: 0x01}), path=[(0x71, 3)])
    return bus

## `scan_check`
One scan per segment (bus + path), each inside the bus lock: select the path, scan, and release
it innermost first even if the scan fails, so no mux is left switched on.

In [ ]:
%%extract ../src/micropython_i2c/manager.py -a
def scan_check(table, buses, locks=None, verbose=True):
    """Confirm every declared address ACKs on its segment.

    Groups entries by segment (bus + path), selects each path, scans once, then releases the
    path innermost first. An ACK means *something* answers at that address, not that it is the
    right part; identity checks confirm the chip.

    Args:
        table: A ``RoutingTable``.
        buses: ``{bus_id: machine.I2C}``.
        locks: Optional ``{bus_id: lock}``; pass the manager's when other code may be using the
            bus concurrently.
        verbose: Print one line per target and a summary.

    Returns:
        ``(all_present, results)`` where each result is a dict with ``bus``, ``hops``, ``addr``,
        ``label`` and ``present``.
    """
    results = []
    for (bus_id, path), members in table.segments().items():
        hops = table.resolve_path(path)
        i2c = buses[bus_id]
        lock = locks[bus_id] if locks else default_lock()
        with lock:
            path_select(i2c, hops)
            try:
                found = set(i2c.scan())
            finally:
                path_release(i2c, hops)
        for addr, label in members:
            results.append(
                {
                    "bus": bus_id,
                    "hops": hops,
                    "addr": addr,
                    "label": label,
                    "present": addr in found,
                }  # fmt: skip
            )

    all_ok = all(r["present"] for r in results)
    if verbose:
        for r in results:
            via = "".join(" via {:#04x}:ch{}".format(a, c) for a, c in r["hops"])
            print(
                "[{}] bus{} {:#04x} {}{}".format(
                    "OK  " if r["present"] else "MISS", r["bus"], r["addr"], r["label"], via
                )
            )
        print("scan_check:", "all present" if all_ok else "MISSING targets")
    return all_ok, results

### validation/test

#### validation: nominal usage

In [ ]:
table = RoutingTable.loads(config()).validate(REG)
bus = fake_topology()
ok, results = scan_check(table, {0: bus})
print("mux left deselected:", bus.nodes[0][2].state == 0)

#### validation: anti-nominal usage
A missing part is not an error: it comes back as `present: False` and `all_present` is
`False`. Here the config expects the TMAG5273 at `0x36`, where nothing answers.

In [ ]:
cfg = config()
cfg["devices"][1]["address"] = "0x36"
ok, results = scan_check(RoutingTable.loads(cfg).validate(), {0: fake_topology()})
print("all present:", ok, "| missing:", [r["label"] for r in results if not r["present"]])

#### unit

In [ ]:
%%extract ../tests/test_manager.py -a --strip-ipytest
%%ipytest


def test_scan_check_finds_everything_and_releases_mux():
    bus = fake_topology()
    table = RoutingTable.loads(config()).validate(REG)
    ok, results = scan_check(table, {0: bus}, verbose=False)
    assert ok and len(results) == 3
    assert bus.nodes[0][2].state == 0  # mux left deselected

In [ ]:
%%extract ../tests/test_manager.py -a --strip-ipytest
%%ipytest


def test_scan_check_reports_missing():
    bus = fake_topology()
    cfg = config()
    cfg["devices"][1]["address"] = "0x36"
    table = RoutingTable.loads(cfg).validate()
    ok, results = scan_check(table, {0: bus}, verbose=False)
    assert not ok
    assert [r["label"] for r in results if not r["present"]] == ["hall"]

## `I2CManager`
Holds the table, the registry and the board, and builds in stages so a notebook can stop
between them (scan before any driver touches a part). `bus_factory` replaces `machine.I2C`:
it gets the bus id, that bus's board entry and the clock worked out for it, and returns
anything with the `machine.I2C` methods. Each stage builds only once: `build()` skips what is
already built, so a test can also set `buses` by hand first.

In [ ]:
%%extract ../src/micropython_i2c/manager.py -a
class I2CManager:
    """Owns the buses, per-bus locks, muxes and devices described by a routing table.

    Args:
        table: A validated ``RoutingTable``.
        registry: ``{driver name: class}``, e.g. ``devices.registry()``.
        board: A ``boards.BOARDS`` key or a board dict.
        bus_factory: Optional ``bus_factory(bus_id, bus_cfg, freq)`` returning the bus object
            for ``bus_id`` (e.g. a ``FakeBus`` on the host). ``None`` builds ``machine.I2C``.

    Attributes:
        buses: ``{bus_id: machine.I2C}`` (or whatever ``bus_factory`` returned).
        locks: ``{bus_id: lock}``, one shared by everything on that bus.
        muxes: ``{name: I2CMux}``.
        devices: ``{name: I2CDevice}``.
    """

    def __init__(self, table, registry, board, bus_factory=None):
        self.table = table
        self.registry = registry
        self.board = board_get(board)
        self.bus_factory = bus_factory
        self.buses = {}
        self.locks = {}
        self.muxes = {}
        self.devices = {}

    def __getitem__(self, name):
        return self.get(name)

    def get(self, name):
        """A device or mux by config name.

        Raises:
            KeyError: If no device or mux has that name.
        """
        if name in self.devices:
            return self.devices[name]
        return self.muxes[name]

    def bus_freq(self, bus_id):
        """Clock for ``bus_id``: the lowest ``BAUD_MAX`` of the board and every part on it.

        Raises:
            ValueError: If some part's ``BAUD_MIN`` is above that ceiling.
        """
        ceiling = self.board["buses"][bus_id].get("baud_max", 400_000)
        floor = 0
        for _, e in self.table.entries():
            if e["bus"] == bus_id and e.get("driver"):
                cls = self.registry[e["driver"]]
                ceiling = min(ceiling, cls.BAUD_MAX)
                floor = max(floor, cls.BAUD_MIN)
        if floor > ceiling:
            raise ValueError(
                "bus {}: no clock satisfies every part ({} Hz min > {} Hz max)".format(
                    bus_id, floor, ceiling
                )
            )
        return ceiling

    def buses_build(self):
        """Create each bus and its lock: ``machine.I2C``, or ``bus_factory``'s result."""
        for bus_id in self.table.buses:
            cfg = self.board["buses"][bus_id]
            freq = self.bus_freq(bus_id)
            if self.bus_factory is not None:
                self.buses[bus_id] = self.bus_factory(bus_id, cfg, freq)
            else:
                from machine import I2C, Pin

                pins = {"scl": Pin(cfg["scl"]), "sda": Pin(cfg["sda"])}
                self.buses[bus_id] = I2C(bus_id, freq=freq, **pins)
            self.locks[bus_id] = default_lock()
        return self

    def scan(self, verbose=True):
        """``scan_check`` against the built buses. Call after ``buses_build()``."""
        return scan_check(self.table, self.buses, self.locks, verbose)

    def muxes_build(self):
        """Instantiate every mux on its bus, with the shared bus lock."""
        for name, m in self.table.muxes.items():
            self.muxes[name] = self.registry[m["driver"]](
                bus=self.buses[m["bus"]],
                address=m["address"],
                lock=self.locks[m["bus"]],
                path=self.table.resolve_path(m["path"]),
                reset_pin=m.get("reset_pin"),
            )
        return self

    def devices_build(self):
        """Build each device's connection, then the driver instance."""
        for d in self.table.devices:
            conn = I2CConnection(
                self.buses[d["bus"]],
                d["address"],
                self.table.resolve_path(d["path"]),
                self.locks[d["bus"]],
            )
            self.devices[d["name"]] = self.registry[d["driver"]](conn, **d["init_kwargs"])
        return self

    def build(self):
        """Build whatever is not built yet: buses, then muxes, then devices."""
        if not self.buses:
            self.buses_build()
        if not self.muxes:
            self.muxes_build()
        if not self.devices:
            self.devices_build()
        return self

### validation/test

#### validation: nominal usage
Build through a `bus_factory` that hands back the fake topology: the manager works out the bus
clock (the TCA9548A and the board both cap at 400 kHz), then builds the mux and both drivers,
whose identity reads go through the mux.

In [ ]:
made = []


def factory(bus_id, cfg, freq):
    made.append((bus_id, freq))
    return fake_topology()


mgr = I2CManager(RoutingTable.loads(config()).validate(REG), REG, "feathers3", bus_factory=factory)
mgr.build()
print("factory called with:", made)
print("hall identify:", mgr["hall"].identify(), "| humidity identify:", mgr["humidity"].identify())
mgr["mux_a"].channel_select(2)
print("mux control register after selecting channel 2:", hex(mgr["mux_a"].control_read()))

#### validation: anti-nominal usage

In [ ]:
try:
    mgr["nope"]  # no device or mux of that name
except KeyError as err:
    print("raised KeyError, as expected:\n    {}".format(err))
else:
    raise AssertionError("mgr['nope'] should have raised KeyError")

In [ ]:
slow = {"buses": {0: {"scl": 0, "sda": 0, "baud_max": 5_000}}}  # below every part's BAUD_MIN
try:
    I2CManager(RoutingTable.loads(config()).validate(REG), REG, slow).bus_freq(0)
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("a 5 kHz bus cap should have raised ValueError")

In [ ]:
try:
    I2CManager(RoutingTable.loads(config()).validate(REG), REG, "feathers3").buses_build()  # no factory, no machine
except ImportError as err:
    print("raised ImportError, as expected (the host has no machine module):\n    {}".format(err))
else:
    raise AssertionError("buses_build without a factory should have raised ImportError on the host")

#### unit

In [ ]:
%%extract ../tests/test_manager.py -a --strip-ipytest
%%ipytest


def test_manager_builds_and_talks():
    bus = fake_topology()
    table = RoutingTable.loads(config()).validate(REG)
    mgr = I2CManager(table, REG, "feathers3")
    mgr.buses, mgr.locks = {0: bus}, {0: NullLock()}  # stand in for buses_build()
    mgr.build()
    assert mgr["hall"].identify() == (0x5449, 1)
    assert mgr["humidity"].identify()
    mgr["mux_a"].channel_select(2)
    assert mgr["mux_a"].control_read() == 0x04
    with pytest.raises(KeyError):
        mgr["nope"]

In [ ]:
%%extract ../tests/test_manager.py -a --strip-ipytest
%%ipytest


def test_bus_factory_builds_each_bus_at_its_clock():
    calls = []

    def factory(bus_id, cfg, freq):
        calls.append((bus_id, cfg["scl"], freq))
        return fake_topology()

    mgr = I2CManager(RoutingTable.loads(config()).validate(REG), REG, "feathers3", factory)
    mgr.build()
    assert calls == [(0, 9, 400_000)]
    assert set(mgr.locks) == {0}
    assert mgr["hall"].identify() == (0x5449, 1)

In [ ]:
%%extract ../tests/test_manager.py -a --strip-ipytest
%%ipytest


def test_build_skips_stages_already_built():
    mgr = I2CManager(RoutingTable.loads(config()).validate(REG), REG, "feathers3", lambda *a: 1 / 0)
    mgr.buses, mgr.locks = {0: fake_topology()}, {0: NullLock()}
    mgr.build()  # the factory would raise if buses_build ran again
    assert set(mgr.devices) == {"humidity", "hall"}

In [ ]:
%%extract ../tests/test_manager.py -a --strip-ipytest
%%ipytest


def test_bus_freq_is_lowest_ceiling():
    table = RoutingTable.loads(config()).validate(REG)
    mgr = I2CManager(table, REG, "feathers3")
    assert mgr.bus_freq(0) == 400_000  # TCA9548A and the board both cap at 400 kHz

In [ ]:
%%extract ../tests/test_manager.py -a --strip-ipytest
%%ipytest


def test_bus_freq_refuses_a_cap_below_every_part():
    slow = {"buses": {0: {"scl": 0, "sda": 0, "baud_max": 5_000}}}
    mgr = I2CManager(RoutingTable.loads(config()).validate(REG), REG, slow)
    with pytest.raises(ValueError, match="no clock satisfies every part"):
        mgr.bus_freq(0)

In [ ]:
%%extract ../tests/test_manager.py -a --strip-ipytest
%%ipytest


def test_without_a_factory_the_host_cannot_build_buses():
    mgr = I2CManager(RoutingTable.loads(config()).validate(REG), REG, "feathers3")
    with pytest.raises(ImportError):
        mgr.buses_build()

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package, builds the mux and two sensors on I2C bus 1, and reads them. Every transfer is a
> scan, an identity read or a temperature read: nothing is reconfigured beyond what each
> driver's constructor does.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

Layout: connect and set up, the **matching tests** on the real bench, then the tally and
disconnect.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.debug import check, check_raises, check_reset, check_summary
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.devices import registry
from micropython_i2c.manager import I2CManager
from micropython_i2c.routing import RoutingTable

check_reset()

Notebook-only: the bench as wired (README "The bench", 2026-11-14), the part of it on bus 1
this notebook uses. If the bench is rewired, change this table.

**What to do:** nothing.

In [ ]:
HW_REG = registry(["TCA9548A", "TMP102", "TMAG5273"])
HW_CONFIG = {
    "buses": [1],
    "muxes": {"mux": {"bus": 1, "address": "0x70", "driver": "TCA9548A"}},
    "devices": [
        {"name": "hall", "bus": 1, "address": "0x22", "driver": "TMAG5273", "path": [["mux", 0]]},
        {"name": "temp", "bus": 1, "address": "0x48", "driver": "TMP102", "path": [["mux", 2]]},
    ],
}
hw_table = RoutingTable.loads(HW_CONFIG).validate(HW_REG)

## Matching tests

**Matches** `test_bus_freq_is_lowest_ceiling`, on the real board entry.

**What to do:** nothing.

In [ ]:
hw = I2CManager(hw_table, HW_REG, "feathers3")
check("test_bus_freq_is_lowest_ceiling", hw.bus_freq(1) == 400_000, "{} Hz".format(hw.bus_freq(1)))

**Matches** `test_scan_check_finds_everything_and_releases_mux`: build the real bus, then scan
each segment. The mux must be left with every channel off.

**What to do:** nothing. A `MISS` line names the part that did not answer: check its cable.

In [ ]:
hw.buses_build()
ok, _ = hw.scan()
check("test_scan_check_finds_everything", ok)
check("scan_check releases the mux", hw.buses[1].readfrom(0x70, 1)[0] == 0)

**Matches** `test_manager_builds_and_talks`: build the mux and both drivers (each checks the
part's identity), read each once, and check an unknown name raises `KeyError`.

**What to do:** nothing.

In [ ]:
hw.build()
t = hw["temp"].temperature()
check("TMP102 reads a room temperature", 10.0 < t < 40.0, "{:.2f} C".format(t))
ident = hw["hall"].identify()
check("TMAG5273 identifies through the mux", bool(ident), repr(ident))
check_raises("unknown name raises KeyError", KeyError, hw.get, "nope")

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))